# COVID-19 therapeutics supply chain — NSF EAGER #2028612

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sear-labs/covid-optsc-ffutr-2021/blob/main/notebooks/00_walkthrough.ipynb)

Minimum-cost network flow with penalties on unmet demand. **This notebook is thin on purpose** —
it imports the package and calls it, so it cannot drift from the code. To read the model, read
`src/covid_sc/model.py`.

All inputs are synthetic: one producer, five depots, ten customers.

## 1. Install

On Colab, install from the repository. Locally, `pip install -e ".[dev]"` once.

In [1]:
# Colab only. Locally you have already run `pip install -e ".[dev]"`.
import sys
if "google.colab" in sys.modules:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "git+https://github.com/sear-labs/covid-optsc-ffutr-2021.git"], check=True)
else:
    print("local environment - skipping install")

local environment - skipping install


## 2. Gurobi

This is a small LP — the **free `pip` licence is enough**, no WLS credentials needed. That is the
difference between this model and the lithium one.

## 3. Load the instance

In [2]:
from covid_sc.model import load_instance
inst = load_instance()
print(f"{len(inst['supply'])} producer, {len(inst['through'])} depots, "
      f"{len(inst['demand'])} customers, {len(inst['cost'])} arcs")
print(f"total demand {sum(inst['demand'].values()):,}  supply {sum(inst['supply'].values()):,}")

1 producer, 5 depots, 10 customers, 55 arcs
total demand 404,000  supply 500,000


## 4. Solve

The original notebook recorded **579000.0**. This should match exactly — it is an LP, so it solves
to proven optimality rather than stopping inside a gap.

In [3]:
from covid_sc.model import solve
res = solve(inst)
print(f"objective {res['objective']:,.2f}   (notebook recorded 579,000.00)")
res["flows"]

Set parameter Username


Set parameter LicenseID to value 2750151


Academic license - for non-commercial use only - expires 2026-12-04


objective 579,000.00   (notebook recorded 579,000.00)


,From,To,Flow,Cost
0,P1,D2,50000.0,25000.0
1,P1,D3,64000.0,64000.0
2,P1,D4,40000.0,8000.0
3,P1,D5,250000.0,50000.0
4,D2,C9,50000.0,50000.0
5,D3,C1,50000.0,50000.0
6,D3,C2,10000.0,5000.0
7,D3,C4,4000.0,4000.0
8,D4,C7,25000.0,37500.0
9,D4,C9,15000.0,22500.0


## 5. Who goes short

Empty here — supply and depot capacity both exceed demand in this instance.

In [4]:
res["deficits"] if len(res["deficits"]) else "no customer is short in this instance"

'no customer is short in this instance'